# Model vs students on hand-labeled unseen data

The two prediction sources (`filtered_events_class_with_predicted.csv` and
`..._students.csv`) disagree on tens of thousands of events whose trigger-word
ground truth is `unknown`. For those there is no label to score against, so we
**hand-labeled a sample**.

`manual_labeled_unknown.csv` is 70 such events — all `unknown` to the trigger words
and where the model and students models predict *different* classes — each assigned a
manual label from the model's allowed 20-class taxonomy, based only on the event
`notes`. This notebook scores both models against those manual labels.

Because every row was picked specifically where the two models disagree, at most one
of them can be right on any given row — so these accuracies are a worst-case probe of
behaviour on ambiguous `unknown` events, not the models' overall accuracy.

In [1]:
import pandas as pd
import plotly.express as px

df = pd.read_csv('manual_labeled_unknown.csv')
df['model_correct'] = df['predicted_class_model'] == df['manual_label']
df['students_correct'] = df['predicted_class_students'] == df['manual_label']
print(f'{len(df)} hand-labeled events')
df.head()

70 hand-labeled events


,event_id_cnty,manual_label,predicted_class_model,predicted_class_students,notes,model_correct,students_correct
0,CYP517,public services,labor rights,policies & politics,"On 28 September 2020, about 50 members of the ...",False,False
1,ITA23103,lgbtq,policies & politics,lgbtq,"On 18 May 2024, a large group of people took p...",False,True
2,ITA7407,labor rights,public services,policies & politics,"On 26 March 2021, food delivery riders rode on...",False,False
3,FRA19654,labor rights,lgbtq,policies & politics,"On 19 October 2022, employees of the Sidel com...",False,False
4,GBR8516,public services,policies & politics,culture,"On 28 April 2025, at the call of Unison and Un...",False,False


## Accuracy against the manual labels

Share of the 70 events where each model's predicted class matches the hand-assigned
label. `neither` is the share where both models were wrong (recall they always
disagree, so they can never both be right).

In [2]:
model_acc = df['model_correct'].mean()
students_acc = df['students_correct'].mean()
neither = (~df['model_correct'] & ~df['students_correct']).mean()

summary = pd.DataFrame({
    'outcome': ['Model correct', 'Students correct', 'Neither correct'],
    'share': [model_acc, students_acc, neither],
    'count': [df['model_correct'].sum(), df['students_correct'].sum(),
              (~df['model_correct'] & ~df['students_correct']).sum()],
})
display(summary)

fig = px.bar(summary, x='outcome', y='share', color='outcome', text=summary['share'].round(3),
             labels={'outcome': '', 'share': 'Share of events'})
fig.update_layout(title={'text': 'Agreement with manual labels (disagreeing unknown events)', 'x': 0.5},
                  plot_bgcolor='white', width=700, height=450,
                  yaxis={'range': [0, 1.05]}, showlegend=False)
fig.show()

,outcome,share,count
0,Model correct,0.100000,7
1,Students correct,0.442857,31
2,Neither correct,0.457143,32


## Where each model wins, by manual label

For each true (manual) class, how many of its events the model got right versus the
students model. Bars to the right of zero are events the students model won; to the
left are events the model won.

In [3]:
by_class = df.groupby('manual_label').agg(
    n=('manual_label', 'size'),
    model=('model_correct', 'sum'),
    students=('students_correct', 'sum'),
).sort_values('n', ascending=False).reset_index()
display(by_class)

melted = by_class.melt(id_vars=['manual_label', 'n'], value_vars=['model', 'students'],
                       var_name='source', value_name='correct')
fig = px.bar(melted, x='correct', y='manual_label', color='source', orientation='h',
             barmode='group', labels={'correct': 'Events correct', 'manual_label': 'Manual label',
                                      'source': ''})
fig.update_layout(title={'text': 'Correct predictions per manual label', 'x': 0.5},
                  plot_bgcolor='white', width=850, height=550,
                  yaxis={'categoryorder': 'total ascending'})
fig.show()

,manual_label,n,model,students
0,labor rights,19,4,8
1,policies & politics,18,3,12
2,public services,6,0,1
3,women rights,5,0,3
4,climate,5,0,2
5,unjust law enforcement,4,0,2
6,health care,3,0,1
7,housing,3,0,1
8,environment,3,0,0
9,discrimination,1,0,0


## Per-event detail

Every event with the manual label, both predictions, and who (if anyone) was right.

In [4]:
def winner(r):
    if r['model_correct']:
        return 'model'
    if r['students_correct']:
        return 'students'
    return 'neither'

detail = df.copy()
detail['correct'] = detail.apply(winner, axis=1)
detail = detail[['event_id_cnty', 'manual_label', 'predicted_class_model',
                 'predicted_class_students', 'correct', 'notes']]
with pd.option_context('display.max_colwidth', 90, 'display.max_rows', 100):
    display(detail)

,event_id_cnty,manual_label,predicted_class_model,predicted_class_students,correct,notes
0,CYP517,public services,labor rights,policies & politics,neither,"On 28 September 2020, about 50 members of the Turkish Cypriot electricity union EL-SEN..."
1,ITA23103,lgbtq,policies & politics,lgbtq,students,"On 18 May 2024, a large group of people took part in the pride parade in Parma (Emilia..."
2,ITA7407,labor rights,public services,policies & politics,neither,"On 26 March 2021, food delivery riders rode on their bikes through the streets and lat..."
3,FRA19654,labor rights,lgbtq,policies & politics,neither,"On 19 October 2022, employees of the Sidel company gathered for a picket, as a part of..."
4,GBR8516,public services,policies & politics,culture,neither,"On 28 April 2025, at the call of Unison and Unite, campaigners gathered outside the Co..."
5,FRA27350,labor rights,environment,lgbtq,neither,"On 1 September 2023, people, including Valdunes employees and CGT affiliates, demonstr..."
6,ITA17129,labor rights,animal welfare,education,neither,"On 28 September 2022, nine former Keller workers climbed onto the rooftop of the Anas ..."
7,ESP2375,unjust law enforcement,labor rights,women rights,neither,"On 27 October 2020, citizens from Nigeria, predominantly women, gathered in Villabona ..."
8,AUT609,labor rights,public services,education,neither,"On 1 May 2021, between 400 to 600 people protested in Salzburg on the occasion of May ..."
9,LVA257,public services,policies & politics,public services,students,"On 30 April 2024, residents of Roja with about 250 cars drove through the Roja-Valdema..."
